<a href="https://colab.research.google.com/github/GautamAjesh/A-Comparative-Analysis-of-Quantization-Methods-Across-NLP-Tasks-in-Small-Language-Models/blob/main/final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
# ---- Cell 1: Setup ----

!pip install -q transformers datasets statsmodels scipy torch

import os, re, json, time, string, random, warnings
import numpy as np
import torch
from collections import Counter
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForQuestionAnswering,
)
from datasets import load_dataset

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("Torch:", torch.__version__)

QUICK_MODE = False   # set True for a ~30-min smoke run
N_SENT = 500
N_QA   = 500
N_RTE  = None        # None = use full 277-example validation set
LATENCY_RUNS = 20 if QUICK_MODE else 50

RESULTS = {}

Device: cpu
Torch: 2.11.0+cpu


In [5]:
# ============================================================
# ---- Cell 2: ALL helper functions defined BEFORE any use ----
# ============================================================

def quantize_int8(model):
    model.eval()
    return torch.quantization.quantize_dynamic(
        model, {torch.nn.Linear}, dtype=torch.qint8
    )

def get_model_size_mb(model):
    path = "temp_size_check.pt"
    torch.save(model.state_dict(), path)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    os.remove(path)
    return size_mb

def normalize_text(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in string.punctuation)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def compute_f1(pred, truth):
    p = normalize_text(pred).split()
    t = normalize_text(truth).split()
    if not p or not t:
        return int(p == t)
    common = Counter(p) & Counter(t)
    n = sum(common.values())
    if n == 0:
        return 0
    prec = n / len(p); rec = n / len(t)
    return 2 * prec * rec / (prec + rec)

def qa_answer(model, tok, question, context):
    inp = tok(question, context, return_tensors="pt",
              truncation=True, max_length=384).to(DEVICE)
    with torch.no_grad():
        out = model(**inp)
    s = int(torch.argmax(out.start_logits))
    e = int(torch.argmax(out.end_logits))
    if e < s:
        s, e = e, s
    return tok.decode(inp["input_ids"][0][s:e+1], skip_special_tokens=True)

def probe_qa_checkpoint(name, n=30):
    """Returns (mean_f1, empty_frac, tok, model) or raises."""
    tok = AutoTokenizer.from_pretrained(name)
    mdl = AutoModelForQuestionAnswering.from_pretrained(name).to(DEVICE).eval()
    ds = load_dataset("rajpurkar/squad", split="validation").shuffle(seed=SEED).select(range(n))
    f1s, empties = [], 0
    for ex in ds:
        ans = qa_answer(mdl, tok, ex["question"], ex["context"])
        if not ans.strip():
            empties += 1
        f1s.append(max(compute_f1(ans, a) for a in ex["answers"]["text"]))
    return float(np.mean(f1s)), empties / n, tok, mdl

def probe_classifier(name, dataset, tok_fn, n=50):
    """Returns (accuracy, Counter(preds), tok, model)."""
    tok = AutoTokenizer.from_pretrained(name)
    mdl = AutoModelForSequenceClassification.from_pretrained(name).to(DEVICE).eval()
    sub = dataset.select(range(min(n, len(dataset))))
    preds = []
    for ex in sub:
        inp = tok_fn(tok, ex)
        with torch.no_grad():
            out = mdl(**inp)
        preds.append(int(torch.argmax(out.logits, dim=1).item()))
    acc = float(np.mean([p == ex["label"] for p, ex in zip(preds, sub)]))
    return acc, Counter(preds), tok, mdl

def eval_sentiment(model, tok, ds):
    preds = []
    for ex in ds:
        inp = tok(ex["sentence"], return_tensors="pt", truncation=True).to(DEVICE)
        with torch.no_grad():
            out = model(**inp)
        preds.append(int(torch.argmax(out.logits, dim=1).item()))
    labels = [ex["label"] for ex in ds]
    acc = float(np.mean([p == l for p, l in zip(preds, labels)]))
    return acc, preds, labels

def eval_nli(model, tok, ds):
    preds = []
    for ex in ds:
        inp = tok(ex["sentence1"], ex["sentence2"], return_tensors="pt",
                  truncation=True).to(DEVICE)
        with torch.no_grad():
            out = model(**inp)
        preds.append(int(torch.argmax(out.logits, dim=1).item()))
    labels = [ex["label"] for ex in ds]
    acc = float(np.mean([p == l for p, l in zip(preds, labels)]))
    return acc, preds, labels

def eval_qa(model, tok, ds):
    f1s = []
    for ex in ds:
        ans = qa_answer(model, tok, ex["question"], ex["context"])
        f1s.append(max(compute_f1(ans, a) for a in ex["answers"]["text"]))
    return float(np.mean(f1s)), f1s

def mcnemar_report(fp32_preds, int8_preds, labels):
    fc = np.array(fp32_preds) == np.array(labels)
    ic = np.array(int8_preds)  == np.array(labels)
    both_ok   = int(np.sum( fc &  ic))
    fp32_only = int(np.sum( fc & ~ic))
    int8_only = int(np.sum(~fc &  ic))
    both_bad  = int(np.sum(~fc & ~ic))
    table = [[both_ok, fp32_only], [int8_only, both_bad]]
    exact = (fp32_only + int8_only) < 25
    res = mcnemar(table, exact=exact)
    return {"table": table, "statistic": float(res.statistic),
            "p_value": float(res.pvalue), "exact": exact}

def paired_t_report(fp32_scores, int8_scores):
    t, p = stats.ttest_rel(fp32_scores, int8_scores)
    diff = np.array(fp32_scores) - np.array(int8_scores)
    return {"t": float(t), "p_value": float(p),
            "mean_diff": float(np.mean(diff)),
            "ci95": [float(x) for x in stats.t.interval(
                0.95, len(diff)-1, loc=np.mean(diff), scale=stats.sem(diff))]}

def measure_latency(fn, n=LATENCY_RUNS):
    for _ in range(3):
        fn()
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(n):
        fn()
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - start) / n * 1000

print("Helpers defined.")

Helpers defined.


In [6]:
# ---- Cell 3: Datasets (random, seed=42) ----
print("\n=== Datasets ===")
sent_ds     = load_dataset("stanfordnlp/sst2",       split="validation").shuffle(seed=SEED).select(range(N_SENT))
rte_ds_full = load_dataset("nyu-mll/glue", "rte",    split="validation").shuffle(seed=SEED)
rte_ds      = rte_ds_full if N_RTE is None else rte_ds_full.select(range(N_RTE))
qa_ds       = load_dataset("rajpurkar/squad",        split="validation").shuffle(seed=SEED).select(range(N_QA))
print(f"SST-2: {len(sent_ds)}  |  RTE: {len(rte_ds)}  |  SQuAD: {len(qa_ds)}")



=== Datasets ===


README.md:   0%|          | 0.00/5.27k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

...

rte/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2490 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/277 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3000 [00:00<?, ? examples/s]

SST-2: 500  |  RTE: 277  |  SQuAD: 500


In [7]:
# ---- Cell 4: DistilBERT models ----
print("\n=== DistilBERT ===")
d_sent_tok  = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")
d_sent_fp32 = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased-finetuned-sst-2-english").to(DEVICE).eval()

d_nli_tok  = AutoTokenizer.from_pretrained("textattack/distilbert-base-uncased-RTE")
d_nli_fp32 = AutoModelForSequenceClassification.from_pretrained(
    "textattack/distilbert-base-uncased-RTE").to(DEVICE).eval()

d_qa_tok  = AutoTokenizer.from_pretrained("distilbert-base-uncased-distilled-squad")
d_qa_fp32 = AutoModelForQuestionAnswering.from_pretrained(
    "distilbert-base-uncased-distilled-squad").to(DEVICE).eval()

d_sent_int8 = quantize_int8(d_sent_fp32)
d_nli_int8  = quantize_int8(d_nli_fp32)
d_qa_int8   = quantize_int8(d_qa_fp32)
print("DistilBERT ready.")


=== DistilBERT ===


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DistilBERT ready.


In [8]:
# ---- Cell 5: ALBERT sentiment ----
print("\n=== ALBERT sentiment ===")
a_sent_tok  = AutoTokenizer.from_pretrained("textattack/albert-base-v2-SST-2")
a_sent_fp32 = AutoModelForSequenceClassification.from_pretrained(
    "textattack/albert-base-v2-SST-2").to(DEVICE).eval()
a_sent_int8 = quantize_int8(a_sent_fp32)
print("ALBERT sentiment ready.")


=== ALBERT sentiment ===


Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]

ALBERT sentiment ready.


In [9]:
# ---- Cell 6: ALBERT QA (probe the primary; no fallback) ----
print("\n=== ALBERT QA ===")
QA_PRIMARY = "Firat/albert-base-v2-finetuned-squad"
QA_CHECKPOINT_USED = None
a_qa_tok = a_qa_fp32 = a_qa_int8 = None
qa_probe = None

try:
    qa_f1_probe, qa_empty_frac, a_qa_tok, a_qa_fp32 = probe_qa_checkpoint(QA_PRIMARY, n=30)
    qa_probe = {"checkpoint": QA_PRIMARY, "probe_f1": qa_f1_probe,
                "empty_frac": qa_empty_frac}
    print(f"  {QA_PRIMARY}: probe F1 = {qa_f1_probe*100:.2f} | empty = {qa_empty_frac*100:.1f}%")
    if qa_f1_probe < 0.10:
        print("  --> Below 0.10 threshold. ALBERT-QA will be EXCLUDED from the paper.")
        a_qa_tok = a_qa_fp32 = None
    else:
        a_qa_int8 = quantize_int8(a_qa_fp32)
        QA_CHECKPOINT_USED = QA_PRIMARY
        print("  --> Accepted.")
except Exception as exc:
    print(f"  {QA_PRIMARY}: FAILED ({exc})")
    qa_probe = {"checkpoint": QA_PRIMARY, "error": str(exc)}
    a_qa_tok = a_qa_fp32 = a_qa_int8 = None


=== ALBERT QA ===


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

  Firat/albert-base-v2-finetuned-squad: probe F1 = 71.70 | empty = 6.7%
  --> Accepted.


In [10]:
# ---- Cell 7: ALBERT NLI — verify all three candidates ----
print("\n=== ALBERT NLI verification ===")
NLI_CANDIDATES = [
    "anirudh21/albert-base-v2-finetuned-rte",
    "Alireza1044/albert-base-v2-rte",
    "textattack/albert-base-v2-RTE",
]

def rte_tok_fn(tok, ex):
    return tok(ex["sentence1"], ex["sentence2"], return_tensors="pt",
               truncation=True).to(DEVICE)

nli_verification = {}
a_nli_tok = a_nli_fp32 = a_nli_int8 = None
for name in NLI_CANDIDATES:
    try:
        acc, dist, tok, mdl = probe_classifier(name, rte_ds_full, rte_tok_fn, n=50)
        distinct = len(dist)
        status = "VERIFIED" if distinct > 1 else "CONSTANT-CLASS (rejected)"
        print(f"  {name}: acc={acc*100:.1f}%  preds={dict(dist)}  -> {status}")
        nli_verification[name] = {
            "accuracy_on_50": acc,
            "prediction_distribution": {str(k): v for k, v in dist.items()},
            "distinct_classes": distinct,
            "status": status,
        }
        if distinct > 1 and a_nli_fp32 is None:
            a_nli_tok, a_nli_fp32 = tok, mdl
    except Exception as exc:
        print(f"  {name}: FAILED ({exc})")
        nli_verification[name] = {"error": str(exc)}

if a_nli_fp32 is not None:
    a_nli_int8 = quantize_int8(a_nli_fp32)
    print("ALBERT NLI: verified checkpoint found and quantized.")
else:
    print("ALBERT NLI: EXCLUDED (no verified checkpoint).")



=== ALBERT NLI verification ===


config.json:   0%|          | 0.00/889 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 46.7MB            

model.safetensors: downloading bytes:           |  0.00B            

  textattack/albert-base-v2-RTE: acc=48.0%  preds={1: 50}  -> CONSTANT-CLASS (rejected)
ALBERT NLI: verified checkpoint found and quantized.


In [11]:
# ---- Cell 8: Run all evaluations ----
print("\n=== Evaluations ===")

d_sent_fp32_acc, d_sent_fp32_preds, sent_labels = eval_sentiment(d_sent_fp32, d_sent_tok, sent_ds)
d_sent_int8_acc, d_sent_int8_preds, _           = eval_sentiment(d_sent_int8, d_sent_tok, sent_ds)

d_nli_fp32_acc, d_nli_fp32_preds, nli_labels = eval_nli(d_nli_fp32, d_nli_tok, rte_ds)
d_nli_int8_acc, d_nli_int8_preds, _          = eval_nli(d_nli_int8, d_nli_tok, rte_ds)

d_qa_fp32_f1, d_qa_fp32_scores = eval_qa(d_qa_fp32, d_qa_tok, qa_ds)
d_qa_int8_f1, d_qa_int8_scores = eval_qa(d_qa_int8, d_qa_tok, qa_ds)

a_sent_fp32_acc, a_sent_fp32_preds, _ = eval_sentiment(a_sent_fp32, a_sent_tok, sent_ds)
a_sent_int8_acc, a_sent_int8_preds, _ = eval_sentiment(a_sent_int8, a_sent_tok, sent_ds)

if a_qa_fp32 is not None:
    a_qa_fp32_f1, a_qa_fp32_scores = eval_qa(a_qa_fp32, a_qa_tok, qa_ds)
    a_qa_int8_f1, a_qa_int8_scores = eval_qa(a_qa_int8, a_qa_tok, qa_ds)
else:
    a_qa_fp32_f1 = a_qa_int8_f1 = None
    a_qa_fp32_scores = a_qa_int8_scores = None

if a_nli_fp32 is not None:
    a_nli_fp32_acc, a_nli_fp32_preds, _ = eval_nli(a_nli_fp32, a_nli_tok, rte_ds)
    a_nli_int8_acc, a_nli_int8_preds, _ = eval_nli(a_nli_int8, a_nli_tok, rte_ds)
else:
    a_nli_fp32_acc = a_nli_int8_acc = None

print("Done.")



=== Evaluations ===
Done.


In [12]:
# ---- Cell 9: Statistics ----
print("\n=== Statistics ===")
stats_out = {
    "distilbert_sentiment": mcnemar_report(d_sent_fp32_preds, d_sent_int8_preds, sent_labels),
    "distilbert_nli":       mcnemar_report(d_nli_fp32_preds, d_nli_int8_preds, nli_labels),
    "distilbert_qa":        paired_t_report(d_qa_fp32_scores, d_qa_int8_scores),
    "albert_sentiment":     mcnemar_report(a_sent_fp32_preds, a_sent_int8_preds, sent_labels),
}
if a_qa_fp32 is not None:
    stats_out["albert_qa"] = paired_t_report(a_qa_fp32_scores, a_qa_int8_scores)
if a_nli_fp32 is not None:
    stats_out["albert_nli"] = mcnemar_report(a_nli_fp32_preds, a_nli_int8_preds, nli_labels)

for k, v in stats_out.items():
    print(f"{k}: {v}")



=== Statistics ===
distilbert_sentiment: {'table': [[439, 15], [10, 36]], 'statistic': 0.64, 'p_value': 0.4237107971667936, 'exact': False}
distilbert_nli: {'table': [[157, 23], [11, 86]], 'statistic': 3.5588235294117645, 'p_value': 0.0592297037272794, 'exact': False}
distilbert_qa: {'t': 4.313525993103351, 'p_value': 1.9375034714595124e-05, 'mean_diff': 0.04982034895785935, 'ci95': [0.027128123612916888, 0.07251257430280181]}
albert_sentiment: {'table': [[224, 236], [15, 25]], 'statistic': 192.82868525896416, 'p_value': 7.672086853854697e-44, 'exact': False}
albert_qa: {'t': 34.99541773197968, 'p_value': 2.0264550210060422e-136, 'mean_diff': 0.664578822024425, 'ci95': [0.6272677216323828, 0.7018899224164672]}
albert_nli: {'table': [[78, 56], [54, 89]], 'statistic': 0.00909090909090909, 'p_value': 0.924039800681422, 'exact': False}


In [13]:
# ---- Cell 10: Latency & size ----
print("\n=== Latency & size ===")
sent_inp = d_sent_tok(sent_ds[0]["sentence"], return_tensors="pt").to(DEVICE)
rte_inp  = d_nli_tok(rte_ds[0]["sentence1"], rte_ds[0]["sentence2"],
                     return_tensors="pt", truncation=True).to(DEVICE)
qa_inp   = d_qa_tok(qa_ds[0]["question"], qa_ds[0]["context"],
                    return_tensors="pt", truncation=True, max_length=384).to(DEVICE)

latency = {
    "distilbert_sentiment_fp32": measure_latency(lambda: d_sent_fp32(**sent_inp)),
    "distilbert_sentiment_int8": measure_latency(lambda: d_sent_int8(**sent_inp)),
    "distilbert_nli_fp32":       measure_latency(lambda: d_nli_fp32(**rte_inp)),
    "distilbert_nli_int8":       measure_latency(lambda: d_nli_int8(**rte_inp)),
    "distilbert_qa_fp32":        measure_latency(lambda: d_qa_fp32(**qa_inp)),
    "distilbert_qa_int8":        measure_latency(lambda: d_qa_int8(**qa_inp)),
    "albert_sentiment_fp32":     measure_latency(lambda: a_sent_fp32(**sent_inp)),
    "albert_sentiment_int8":     measure_latency(lambda: a_sent_int8(**sent_inp)),
}
if a_qa_fp32 is not None:
    latency["albert_qa_fp32"] = measure_latency(lambda: a_qa_fp32(**qa_inp))
    latency["albert_qa_int8"] = measure_latency(lambda: a_qa_int8(**qa_inp))
if a_nli_fp32 is not None:
    latency["albert_nli_fp32"] = measure_latency(lambda: a_nli_fp32(**rte_inp))
    latency["albert_nli_int8"] = measure_latency(lambda: a_nli_int8(**rte_inp))

sizes = {
    "distilbert_sentiment_fp32": get_model_size_mb(d_sent_fp32),
    "distilbert_sentiment_int8": get_model_size_mb(d_sent_int8),
    "distilbert_nli_fp32":       get_model_size_mb(d_nli_fp32),
    "distilbert_nli_int8":       get_model_size_mb(d_nli_int8),
    "distilbert_qa_fp32":        get_model_size_mb(d_qa_fp32),
    "distilbert_qa_int8":        get_model_size_mb(d_qa_int8),
    "albert_sentiment_fp32":     get_model_size_mb(a_sent_fp32),
    "albert_sentiment_int8":     get_model_size_mb(a_sent_int8),
}
if a_qa_fp32 is not None:
    sizes["albert_qa_fp32"] = get_model_size_mb(a_qa_fp32)
    sizes["albert_qa_int8"] = get_model_size_mb(a_qa_int8)
if a_nli_fp32 is not None:
    sizes["albert_nli_fp32"] = get_model_size_mb(a_nli_fp32)
    sizes["albert_nli_int8"] = get_model_size_mb(a_nli_int8)

print("Latency:", json.dumps(latency, indent=2))
print("Sizes:", json.dumps(sizes, indent=2))



=== Latency & size ===
Latency: {
  "distilbert_sentiment_fp32": 194.9402075799935,
  "distilbert_sentiment_int8": 77.4540509399958,
  "distilbert_nli_fp32": 151.1883828400005,
  "distilbert_nli_int8": 63.217433159998116,
  "distilbert_qa_fp32": 331.3534444800007,
  "distilbert_qa_int8": 180.6657833600002,
  "albert_sentiment_fp32": 198.42785113999525,
  "albert_sentiment_int8": 98.91627396000331,
  "albert_qa_fp32": 997.0741079799973,
  "albert_qa_int8": 449.2778718599948,
  "albert_nli_fp32": 275.05598446000477,
  "albert_nli_int8": 170.90622871999585
}
Sizes: {
  "distilbert_sentiment_fp32": 255.45248317718506,
  "distilbert_sentiment_int8": 132.28995418548584,
  "distilbert_nli_fp32": 255.45248317718506,
  "distilbert_nli_int8": 132.28995418548584,
  "distilbert_qa_fp32": 253.19893169403076,
  "distilbert_qa_int8": 131.723219871521,
  "albert_sentiment_fp32": 44.58651161193848,
  "albert_sentiment_int8": 22.370655059814453,
  "albert_qa_fp32": 42.33290100097656,
  "albert_qa_int8"

In [14]:
# ---- Cell 11: Dump everything ----
RESULTS = {
    "metadata": {
        "seed": SEED, "device": DEVICE, "quick_mode": QUICK_MODE,
        "n_sent": len(sent_ds), "n_rte": len(rte_ds), "n_qa": len(qa_ds),
        "latency_runs": LATENCY_RUNS,
        "albert_qa_probe": qa_probe,
        "albert_qa_checkpoint_used": QA_CHECKPOINT_USED,
        "albert_nli_verification": nli_verification,
    },
    "accuracy_f1": {
        "distilbert_sentiment": {"fp32": d_sent_fp32_acc, "int8": d_sent_int8_acc,
                                 "drop_pts": (d_sent_fp32_acc - d_sent_int8_acc)*100},
        "distilbert_nli":       {"fp32": d_nli_fp32_acc,  "int8": d_nli_int8_acc,
                                 "drop_pts": (d_nli_fp32_acc  - d_nli_int8_acc)*100},
        "distilbert_qa":        {"fp32_f1": d_qa_fp32_f1, "int8_f1": d_qa_int8_f1,
                                 "drop_pts": (d_qa_fp32_f1 - d_qa_int8_f1)*100},
        "albert_sentiment":     {"fp32": a_sent_fp32_acc, "int8": a_sent_int8_acc,
                                 "drop_pts": (a_sent_fp32_acc - a_sent_int8_acc)*100},
        "albert_qa": (
            {"fp32_f1": a_qa_fp32_f1, "int8_f1": a_qa_int8_f1,
             "drop_pts": (a_qa_fp32_f1 - a_qa_int8_f1)*100}
            if a_qa_fp32 is not None else "excluded: probe failed"
        ),
        "albert_nli": (
            {"fp32": a_nli_fp32_acc, "int8": a_nli_int8_acc,
             "drop_pts": (a_nli_fp32_acc - a_nli_int8_acc)*100}
            if a_nli_fp32 is not None else "excluded: no verified checkpoint"
        ),
    },
    "statistics": stats_out,
    "latency_ms": latency,
    "size_mb": sizes,
}
with open("results.json", "w") as f:
    json.dump(RESULTS, f, indent=2, default=str)

print("\n" + "="*70)
print("ALL RESULTS (copy this whole block back)")
print("="*70)
print(json.dumps(RESULTS, indent=2, default=str))



ALL RESULTS (copy this whole block back)
{
  "metadata": {
    "seed": 42,
    "device": "cpu",
    "quick_mode": false,
    "n_sent": 500,
    "n_rte": 277,
    "n_qa": 500,
    "latency_runs": 50,
    "albert_qa_probe": {
      "checkpoint": "Firat/albert-base-v2-finetuned-squad",
      "probe_f1": 0.7169898989898991,
      "empty_frac": 0.06666666666666667
    },
    "albert_qa_checkpoint_used": "Firat/albert-base-v2-finetuned-squad",
    "albert_nli_verification": {
      "anirudh21/albert-base-v2-finetuned-rte": {
        "accuracy_on_50": 0.5,
        "prediction_distribution": {
          "0": 43,
          "1": 7
        },
        "distinct_classes": 2,
        "status": "VERIFIED"
      },
      "Alireza1044/albert-base-v2-rte": {
        "accuracy_on_50": 0.48,
        "prediction_distribution": {
          "1": 50
        },
        "distinct_classes": 1,
        "status": "CONSTANT-CLASS (rejected)"
      },
      "textattack/albert-base-v2-RTE": {
        "accuracy_on_50

In [15]:
# ---- Cell 12: Summary table ----
print("\n" + "="*70)
print("SUMMARY TABLE")
print("="*70)
print(f"{'Model':<12}{'Task':<12}{'FP32':>10}{'INT8':>10}{'Drop':>10}{'n':>6}")
print("-"*60)
rows = [
    ("DistilBERT", "Sentiment", d_sent_fp32_acc, d_sent_int8_acc, len(sent_ds)),
    ("DistilBERT", "NLI",       d_nli_fp32_acc,  d_nli_int8_acc,  len(rte_ds)),
    ("DistilBERT", "QA",        d_qa_fp32_f1,    d_qa_int8_f1,    len(qa_ds)),
    ("ALBERT",     "Sentiment", a_sent_fp32_acc, a_sent_int8_acc, len(sent_ds)),
]
if a_qa_fp32 is not None:
    rows.append(("ALBERT", "QA", a_qa_fp32_f1, a_qa_int8_f1, len(qa_ds)))
if a_nli_fp32 is not None:
    rows.append(("ALBERT", "NLI", a_nli_fp32_acc, a_nli_int8_acc, len(rte_ds)))

for m, t, f32, i8, n in rows:
    print(f"{m:<12}{t:<12}{f32*100:>9.2f}{i8*100:>10.2f}{(f32-i8)*100:>10.2f}{n:>6}")


SUMMARY TABLE
Model       Task              FP32      INT8      Drop     n
------------------------------------------------------------
DistilBERT  Sentiment       90.80     89.80      1.00   500
DistilBERT  NLI             64.98     60.65      4.33   277
DistilBERT  QA              79.14     74.16      4.98   500
ALBERT      Sentiment       92.00     47.80     44.20   500
ALBERT      QA              67.38      0.93     66.46   500
ALBERT      NLI             48.38     47.65      0.72   277
